In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_1393.silver.customers_scd2 (
    customer_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    tier STRING,
    signup_date DATE,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    regexp_replace,
    right,
    to_date,
    lit,
    current_date
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

SOURCE_TABLE = "fk_1393.silver.customers_silver"
TARGET_TABLE = "fk_1393.silver.customers_scd2"

INITIAL_DATE = "2026-03-01"


# ============================================================
# 2. READ SOURCE
# ============================================================

source_df = spark.read.table(SOURCE_TABLE)

# ============================================================
# 3. ADD INITIAL EFFECTIVE DATE
# ============================================================

source_df = (
    source_df
    .withColumn(
        "effective_from",
        col("signup_date")
    )
    .withColumn(
        "effective_to",
        lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

source_df.createOrReplaceTempView("customers_source")


# ============================================================
# 4. INITIAL LOAD
# ============================================================

spark.sql(f"""
MERGE INTO {TARGET_TABLE} AS target
USING customers_source AS source
ON target.customer_id = source.customer_id
   AND target.is_current = TRUE

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    full_name,
    email,
    phone,
    city,
    tier,
    signup_date,
    effective_from,
    effective_to,
    is_current
)
VALUES (
    source.customer_id,
    source.full_name,
    source.email,
    source.phone,
    source.city,
    source.tier,
    source.signup_date,
    source.effective_from,
    source.effective_to,
    source.is_current
)
""")